In [3]:
import pandas as pd

df = pd.read_csv('../data/TradeData.csv', encoding='latin-1')
print(df.shape)
print(df.columns.tolist())
df.head()

(1178, 47)
['typeCode', 'freqCode', 'refPeriodId', 'refYear', 'refMonth', 'period', 'reporterCode', 'reporterISO', 'reporterDesc', 'flowCode', 'flowDesc', 'partnerCode', 'partnerISO', 'partnerDesc', 'partner2Code', 'partner2ISO', 'partner2Desc', 'classificationCode', 'classificationSearchCode', 'isOriginalClassification', 'cmdCode', 'cmdDesc', 'aggrLevel', 'isLeaf', 'customsCode', 'customsDesc', 'mosCode', 'motCode', 'motDesc', 'qtyUnitCode', 'qtyUnitAbbr', 'qty', 'isQtyEstimated', 'altQtyUnitCode', 'altQtyUnitAbbr', 'altQty', 'isAltQtyEstimated', 'netWgt', 'isNetWgtEstimated', 'grossWgt', 'isGrossWgtEstimated', 'cifvalue', 'fobvalue', 'primaryValue', 'legacyEstimationFlag', 'isReported', 'isAggregate']


,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
C,A,20140101,2014,52,2014,764,THA,Thailand,M,Import,...,False,NaN,False,9.680795e+09,NaN,9.680795e+09,0,True,False,NaN
C,A,20140101,2014,52,2014,764,THA,Thailand,M,Import,...,True,NaN,False,6.093800e+04,NaN,6.093800e+04,4,True,False,NaN
C,A,20140101,2014,52,2014,764,THA,Thailand,M,Import,...,False,NaN,False,6.379363e+06,NaN,6.379363e+06,0,True,False,NaN
C,A,20140101,2014,52,2014,764,THA,Thailand,M,Import,...,False,NaN,False,1.413294e+07,NaN,1.413294e+07,0,True,False,NaN
C,A,20140101,2014,52,2014,764,THA,Thailand,M,Import,...,True,NaN,False,2.600000e+01,NaN,2.600000e+01,4,True,False,NaN


In [21]:
total_import = df_clean.groupby('year')['trade_value'].sum()
taiwan_import = df_clean[df_clean['partner'] == 'TWN'].groupby('year')['trade_value'].sum()

dependency = (taiwan_import / total_import * 100).round(2)
dependency_df = pd.DataFrame({
    'total_import_usd': total_import,
    'taiwan_import_usd': taiwan_import,
    'taiwan_dependency_pct': dependency
})

print(dependency_df)

      total_import_usd  taiwan_import_usd  taiwan_dependency_pct
year                                                            
2014                36                NaN                    NaN
2015                 0                NaN                    NaN
2016                 0                NaN                    NaN
2017                 0                NaN                    NaN
2018                 0                NaN                    NaN
2019               308                NaN                    NaN
2020               284                NaN                    NaN
2021               320                NaN                    NaN
2022               310                NaN                    NaN
2023               290                NaN                    NaN
2024               372                NaN                    NaN


In [22]:
top_partners = df_clean.groupby('partner')['trade_value'].sum().reset_index()
top_partners = top_partners.sort_values('trade_value', ascending=False)
print(top_partners.head(10))


             partner  trade_value
75           Germany           36
164        Singapore           36
143      Philippines           36
137  Other Asia, nes           36
95             Japan           36
93             Italy           36
181         Thailand           36
128      Netherlands           34
121          Morocco           34
53           Czechia           34


In [26]:
import pandas as pd
import requests

url = "https://comtradeapi.un.org/public/v1/preview/C/A/HS?reporterCode=764&partnerCode=490&cmdCode=8542&flowCode=M&period=2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024"

response = requests.get(url)
data = response.json()

df_api = pd.DataFrame(data['data'])

# กรองเฉพาะ motCode=0 (total) และ partner2Code=0 (ไม่แยก 2nd partner)
df_taiwan = df_api[(df_api['motCode'] == 0) & (df_api['partner2Code'] == 0)][['refYear', 'primaryValue']].copy()
df_taiwan.columns = ['year', 'taiwan_import_usd']
df_taiwan = df_taiwan.groupby('year')['taiwan_import_usd'].sum().reset_index()

print(df_taiwan)

   year  taiwan_import_usd
0  2014       2.207751e+09
1  2015       2.465590e+09
2  2016       2.280656e+09
3  2017       2.981502e+09
4  2018       3.395167e+09
5  2019       3.429520e+09
6  2020       3.919509e+09
7  2021       4.615399e+09
8  2022       5.908972e+09
9  2023       7.326314e+09


In [1]:
# คำนวณ total import (ทุกประเทศ) รายปี
total_import = df_api[(df_api['flowCode'] == 'M') & 
                       (df_api['partner2Code'] == 0)].groupby('refYear')['primaryValue'].sum().reset_index()
total_import.columns = ['year', 'total_import_usd']

# merge กับ taiwan
dependency = pd.merge(taiwan_yearly, total_import, on='year')

# คำนวณ dependency index
dependency['taiwan_dependency_pct'] = (dependency['taiwan_import_usd'] / dependency['total_import_usd']) * 100

print(dependency)

NameError: name 'df_api' is not defined